<a href="https://colab.research.google.com/github/BryanHinostroza/tarea15-bh/blob/develop/GLAB15_HINOSTROZA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Paso 1: Instalar las librerías necesarias
pip install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 11.0 MB/s e

In [2]:
# Paso 2: Importar las librerías
import chromadb
from sentence_transformers import SentenceTransformer

In [3]:
# Paso 3: Inicializar una base de datos local y crear una colección
# Inicializar el motor de vectores local
client = chromadb.Client()
# Crear colección (equivalente a una tabla)
collection = client.create_collection(name="alumnos")
model = SentenceTransformer('all-MiniLM-L6-v2') # Modelo de embedding

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
# Paso 4: Crear entradas de texto para convertirlas en embeddings y almacenarlas en la base de datos
# Lista de documentos de ejemplo
docs = [
    "Luis es un excelente alumno de ciencia de datos",
    "María tiene interés en la inteligencia artificial",
    "Pedro trabaja en procesamiento de lenguaje natural"
]
# Generar vectores
embeddings = model.encode(docs).tolist()
# Insertar en la colección
collection.add(
    documents=docs,
    embeddings=embeddings,
    ids=["1", "2", "3"]
)

In [10]:
# Paso 5: Colocar un prompt cualquiera y hacer una selección hacia nuestra base de datos vectorial
query = "¿Quién estudia Ciencia de Datos?"
query_embedding = model.encode([query]).tolist()
# Buscar los 2 más cercanos
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)
print(results['documents'])

[['Luis es un excelente alumno de ciencia de datos', 'Pedro trabaja en procesamiento de lenguaje natural']]


In [8]:
# Paso 6: Visualizar cómo el modelo convirtió nuestro texto en vectores
data = collection.get(include=["embeddings", "documents"])
for doc, id, emb in zip(data['documents'], data['ids'], data['embeddings']):
    print(f"ID: {id}")
    print(f"Documento: {doc}")
    print(f"Embedding (primeras 5 dimensiones): {emb[:5]}")
    print("-" * 30)

ID: 1
Documento: Luis es un excelente alumno de ciencia de datos
Embedding (primeras 5 dimensiones): [-0.0515781   0.07000642 -0.04834177 -0.05982044 -0.01550355]
------------------------------
ID: 2
Documento: María tiene interés en la inteligencia artificial
Embedding (primeras 5 dimensiones): [-0.03857211 -0.01005658  0.00841351 -0.03409148 -0.09196933]
------------------------------
ID: 3
Documento: Pedro trabaja en procesamiento de lenguaje natural
Embedding (primeras 5 dimensiones): [ 0.04198885  0.0491558  -0.02113147  0.01458174 -0.09477768]
------------------------------


In [9]:
# Paso 7: Visualizar lo que almacena una base de datos vectorial
data = collection.get()
print(data)

{'ids': ['1', '2', '3'], 'embeddings': None, 'documents': ['Luis es un excelente alumno de ciencia de datos', 'María tiene interés en la inteligencia artificial', 'Pedro trabaja en procesamiento de lenguaje natural'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None, None, None]}


# Caso Práctico

Ahora que ya sabe lo básico, desarrolle lo siguiente:
Cree una base de datos vectorial llamado “Tecsup” y guarde información personal suya y de sus
compañeros y luego cree un prompt para poder capturar las respuestas en base a lo guardado.
Luego, utilice la misma solución pero con otro motor de embedding y compare sus respuestas

In [11]:
# Inicializar una base de datos local y crear una colección llamada "Tecsup"
client = chromadb.Client()
collection_tecsup = client.create_collection(name="Tecsup")
model_minilm = SentenceTransformer('all-MiniLM-L6-v2') # Primer motor de embedding

In [12]:
# Crear entradas de texto con información personal y de compañeros y almacenarlas en la base de datos "Tecsup"
docs_tecsup = [
    "Abraham Huaman es un estudiante de ingeniería de Big Data con interés en inteligencia artificial.",
    "Belén Gutierrez está estudiando ciencia de datos y le gusta el aprendizaje automático.",
    "Matías Moreno se especializa en ciberseguridad y redes.",
    "Kevin Olórtegui es una entusiasta del desarrollo web y la programación front-end.",
    "Mi nombre es Bryan, y soy un asistente de IA enfocado en el procesamiento de lenguaje natural.",
    "Joshua trabaja en procesamiento de lenguaje natural."
]
embeddings_tecsup = model_minilm.encode(docs_tecsup).tolist()

ids_tecsup = [str(i) for i in range(1, len(docs_tecsup) + 1)] # Generar IDs únicos

collection_tecsup.add(
    documents=docs_tecsup,
    embeddings=embeddings_tecsup,
    ids=ids_tecsup
)

In [19]:
# Crear un prompt para capturar respuestas de la base de datos "Tecsup" (usando all-MiniLM-L6-v2)
query_tecsup = "¿Quién está estudiando Ciencia de Datos?"
query_embedding_tecsup = model_minilm.encode([query_tecsup]).tolist()

results_tecsup = collection_tecsup.query(
    query_embeddings=query_embedding_tecsup,
    n_results=2
)

print("Resultados de la consulta (all-MiniLM-L6-v2):")
print(results_tecsup['documents'])

Resultados de la consulta (all-MiniLM-L6-v2):
[['Belén Gutierrez está estudiando ciencia de datos y le gusta el aprendizaje automático.', 'Mi nombre es Bryan, y soy un asistente de IA enfocado en el procesamiento de lenguaje natural.']]


In [14]:
# Utilizar la misma solución pero con otro motor de embedding y comparar las respuestas
model_qa = SentenceTransformer('all-mpnet-base-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
# Volver a generar embeddings para los mismos documentos con el nuevo modelo
embeddings_tecsup_qa = model_qa.encode(docs_tecsup).tolist()

In [16]:
# Crear una nueva colección para la comparación o actualizar la existente si es viable (para simplificar, crearemos una nueva)
# client.delete_collection(name="Tecsup")
collection_tecsup_qa = client.create_collection(name="Tecsup_QA")

collection_tecsup_qa.add(
    documents=docs_tecsup,
    embeddings=embeddings_tecsup_qa,
    ids=ids_tecsup
)

In [23]:
# Realizar la misma consulta con el nuevo motor de embedding
query_embedding_tecsup_qa = model_qa.encode([query_tecsup]).tolist()

results_tecsup_qa = collection_tecsup_qa.query(
    query_embeddings=query_embedding_tecsup_qa,
    n_results=2
)

print("\nResultados de la consulta (all-mpnet-base-v2):")
print(results_tecsup_qa['documents'])



Resultados de la consulta (all-mpnet-base-v2):
[['Belén Gutierrez está estudiando ciencia de datos y le gusta el aprendizaje automático.', 'Mi nombre es Bryan, y soy un asistente de IA enfocado en el procesamiento de lenguaje natural.']]


In [25]:
# Visualizar cómo el modelo convirtió nuestro texto en vectores
data = collection_tecsup_qa.get(include=["embeddings", "documents"])
for doc, id, emb in zip(data['documents'], data['ids'], data['embeddings']):
    print(f"ID: {id}")
    print(f"Documento: {doc}")
    print(f"Embedding (primeras 5 dimensiones): {emb[:5]}")
    print("-" * 30)

ID: 1
Documento: Abraham Huaman es un estudiante de ingeniería de Big Data con interés en inteligencia artificial.
Embedding (primeras 5 dimensiones): [-4.91391635e-04  9.83191952e-02 -4.06288914e-02  2.86790855e-05
 -3.28125358e-02]
------------------------------
ID: 2
Documento: Belén Gutierrez está estudiando ciencia de datos y le gusta el aprendizaje automático.
Embedding (primeras 5 dimensiones): [-0.05657328  0.04378007 -0.04287744 -0.00651564 -0.03849741]
------------------------------
ID: 3
Documento: Matías Moreno se especializa en ciberseguridad y redes.
Embedding (primeras 5 dimensiones): [-0.01970688  0.04901293  0.00619811 -0.01513326 -0.00874524]
------------------------------
ID: 4
Documento: Kevin Olórtegui es una entusiasta del desarrollo web y la programación front-end.
Embedding (primeras 5 dimensiones): [-0.00837162  0.01615354 -0.02861755  0.01369673  0.01249179]
------------------------------
ID: 5
Documento: Mi nombre es Bryan, y soy un asistente de IA enfocado e

# Conclusión

La igualdad de resultados no indica una falta de diferenciación entre los modelos, sino más bien que en este caso de uso particular (conjunto de datos pequeño y consulta directa) no presenta la complejidad necesaria para que las fortalezas únicas de cada modelo se destaquen. Ambos son lo suficientemente buenos para manejar esta tarea de recuperación de información de forma similar. Las diferencias entre modelos de embedding suelen ser más patentes en búsquedas más matizadas, conjuntos de datos más grandes o cuando se evalúa un ranking más profundo de resultados.